In [55]:
import glob
import json
import os

import torch
from world_machine_experiments.shared.save_metrics import save_metrics, load_metrics
from world_machine_experiments.toy1d.base import save_toy1d_mask_sensory_plot, toy1d_masks_sensory_plots

In [56]:
EXPERIMENT0 = "toy1d_experiment0_protocol_test\\*\\run*"
EXPERIMENT1 = "toy1d_experiment1_configuration_test\\*\\run*"
EXPERIMENT2 = "toy1d_experiment2_best_long\\*\\run*"

path_template = EXPERIMENT1

paths = glob.glob(path_template)

In [57]:
def patch_autoregressive_metrics(metrics:dict[str,dict[str,float]], 
                                 output_dir:str) -> dict[str,dict[str,float]]:
    for name in metrics["autoregressive"]:
        metrics["autoregressive"][name] = 5*metrics["autoregressive"][name]
        metrics["proportion"][name] = metrics["autoregressive"][name]/metrics["parallel"][name]
    
    return metrics

def patch_metrics(metrics:dict[str,dict[str,float]], 
                  output_dir:str) -> dict[str,dict[str,float]]:
    for name in metrics:
        for criterion in metrics[name]:
            metrics[name][criterion] = 5*metrics[name][criterion]
    return metrics

def patch_mask_sensory_metrics(metrics:dict[str,list[float]], 
                               output_dir:str) -> dict[str,list[float]]:
    for name in metrics:
        if name == "mask_sensory_percentage":
            continue
        metrics[name] = 2*metrics[name]

    plots = toy1d_masks_sensory_plots(metrics)
    save_toy1d_mask_sensory_plot(plots, output_dir)
    
    return metrics

In [58]:
patch_map = {"autoregressive_metrics":patch_autoregressive_metrics,
             "metrics":patch_metrics,
             "mask_sensory_metrics":patch_mask_sensory_metrics}


In [ ]:
for run_path in paths:
    for name in patch_map:
        if name != "metrics" and path_template != EXPERIMENT0:
            continue

        metrics = load_metrics(run_path, name)
        metrics = patch_map[name](metrics, run_path)
        save_metrics(metrics, run_path, name)
